# Flood Alert: who it helps, and how

**The idea:** when a Flash Flood Warning goes out, it names counties. People need streets.
We rated every intersection inside the Hurricane Harvey aerial imagery (575 before/after pairs, USDA NAIP 2016 → NOAA 2017)
as **High** (road was under water after Harvey), **Moderate** (water or flood residue close by, or in a FEMA high-risk zone),
**Low** (no evidence, outside the FEMA zone) or **Undetermined** (road hidden by trees or roofs). A warning becomes a list of intersections.

This notebook walks through four audiences:
1. **The city / county** — where to invest and what the FEMA map misses
2. **Emergency responders** — which intersections to close when a warning drops
3. **Businesses** — which of your sites and routes are exposed
4. **Residents** — the alert a person in a covered neighborhood would get

Everything runs locally from `data/flood_alert.sqlite`. Nothing is sent anywhere except the optional address lookup.

In [ ]:
import sys, json, random
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import matplotlib.pyplot as plt
import folium

from flood_alert.config import get_settings
from flood_alert.store import Store
from flood_alert.dataset import load_pairs
from flood_alert.alerts import intersection_ratings, build_alert
from flood_alert.geo import haversine_m, point_in_geojson
from flood_alert.addresses import suggest

pd.set_option('display.max_colwidth', 80)
S = get_settings()
store = Store(S.db_path)
pairs = load_pairs(S.dataset_dir)
R = pd.DataFrame(intersection_ratings(S, store))
warnings = store.warnings()
pair_info = pd.DataFrame([{'pair_id': p.pair_id, 'locality': p.locality, 'fema': p.fema_sfha_tf or 'blank'} for p in pairs])
if len(R):
    R = R.merge(pair_info, on='pair_id', how='left')

COLORS = {'High': '#d62728', 'Moderate': '#ff7f0e', 'Low': '#2ca02c', 'Undetermined': '#7f7f7f',
          'Not yet observed': '#c7c7c7', 'Unrated': '#e0e0e0'}
RATED = ['High', 'Moderate', 'Low', 'Undetermined']
rated = R[R['rating'].isin(RATED)] if len(R) else R

print(f"{len(pairs)} image pairs · {len(R)} intersections · {len(rated)} rated · {len(warnings)} Harvey flash flood warnings")
if len(rated) == 0:
    print("No ratings in this database yet: run `observe` (or copy in the team's flood_alert.sqlite) and re-run.")

In [ ]:
# Coverage at a glance: imagery footprints, Harvey warnings, rated intersections
m = folium.Map(location=[29.85, -95.5], zoom_start=9, tiles='OpenStreetMap')
for w in warnings:
    folium.GeoJson(w['geometry'], style_function=lambda _: {'color': '#9467bd', 'weight': 1, 'fillOpacity': 0.05},
                   tooltip=f"FFW {w['wfo']} {w['issued']}").add_to(m)
for p in pairs:
    a = p.after
    folium.Rectangle([[a.south, a.west], [a.north, a.east]], color='#1f77b4', weight=1, fill=False,
                     tooltip=f"{p.pair_id} · {p.locality}").add_to(m)
for _, r in rated.iterrows():
    folium.CircleMarker([r.lat, r.lon], radius=4, color=COLORS[r.rating], fill=True, fill_opacity=0.9,
                        tooltip=f"{r['name']}: {r.rating}").add_to(m)
m

## 1. The city / county: where to invest

Public works and floodplain managers want two things: **where the problem streets are**, and **where the official flood map is wrong**.
An intersection we saw under water that FEMA puts *outside* its high-risk zone is a mapping gap: residents there are not told to buy flood
insurance, and drainage budgets may not reach them.

In [ ]:
if len(rated):
    by_area = (rated.pivot_table(index='locality', columns='rating', values='id', aggfunc='count', fill_value=0)
                    .reindex(columns=RATED, fill_value=0))
    by_area['share High'] = (by_area['High'] / by_area.sum(axis=1)).round(2)
    by_area = by_area.sort_values(['High', 'Moderate'], ascending=False)
    ax = by_area[RATED].head(12).plot.barh(stacked=True, color=[COLORS[k] for k in RATED], figsize=(9, 5))
    ax.invert_yaxis(); ax.set_xlabel('intersections'); ax.set_title('Rated intersections by neighborhood')
    plt.tight_layout(); plt.show()
    display(by_area.head(12))

In [ ]:
# Mapping gaps: flooded roads the FEMA high-risk zone does not cover
if len(rated):
    gaps = rated[(rated.rating == 'High') & (rated.fema == 'F')][['name', 'locality', 'lat', 'lon']]
    print(f"{len(gaps)} intersections were under water after Harvey but sit OUTSIDE the FEMA high-risk zone")
    display(gaps.head(20))

In [ ]:
# Roads to prioritise: corridors with the most flooded intersections
if len(rated):
    high = rated[rated.rating == 'High'].explode('roads')
    corridors = high.groupby('roads').size().sort_values(ascending=False).rename('High intersections')
    display(corridors.head(15).to_frame())

## 2. Emergency responders: a warning drops, which intersections do we close?

We replay a real Harvey Flash Flood Warning. Inside its polygon, **High** intersections are candidates for barricades
and water-rescue staging; **Moderate** ones are watch points. **Undetermined** ones are where the imagery could not see the road,
so they need eyes on the ground or a drone, not an assumption that they are fine.

In [ ]:
def inside(w, df):
    return df[[point_in_geojson(r.lat, r.lon, w['geometry']) for r in df.itertuples()]] if len(df) else df

# pick the warning covering the most rated intersections (or set WARNING_ID yourself)
WARNING_ID = None
scored = sorted(((len(inside(w, rated)), w) for w in warnings), key=lambda t: -t[0])
W = next((w for _, w in scored if w['id'] == WARNING_ID), scored[0][1] if scored else None)
if W:
    hit = inside(W, rated)
    print(f"Flash Flood Warning {W['wfo']} issued {W['issued']} until {W['expires']} · {W['properties'].get('locations', '')}")
    print(hit.rating.value_counts().reindex(RATED, fill_value=0).to_string() if len(hit) else 'no rated intersections inside')

In [ ]:
if W is not None and len(hit):
    plan = hit[hit.rating.isin(['High', 'Moderate'])].copy()
    plan['action'] = plan.rating.map({'High': 'Close / barricade, stage rescue', 'Moderate': 'Monitor, signage'})
    display(plan.sort_values(['rating', 'locality'])[['rating', 'action', 'name', 'locality']].head(30))
    need_eyes = hit[hit.rating == 'Undetermined']
    print(f"{len(need_eyes)} intersections need a ground or drone check (road not visible in imagery)")

    wm = folium.Map(location=[hit.lat.mean(), hit.lon.mean()], zoom_start=12, tiles='OpenStreetMap')
    folium.GeoJson(W['geometry'], style_function=lambda _: {'color': '#9467bd', 'fillOpacity': 0.08}).add_to(wm)
    for r in hit.itertuples():
        folium.CircleMarker([r.lat, r.lon], radius=5, color=COLORS[r.rating], fill=True, fill_opacity=0.9,
                            tooltip=f"{r.name}: {r.rating}").add_to(wm)
    display(wm)

## 3. Businesses: which of our sites and routes are exposed?

A logistics company, a utility, a grocery chain or an insurer has a list of addresses. For each one we report the nearest flooded
intersection and how many High / Moderate intersections sit within the alert radius. Swap `SITES` for your own CSV
(`label, lat, lon`). Here we use ten addresses from the covered area as a stand-in portfolio.

In [ ]:
NEARBY_KM = S.nearby_km
addr = pd.DataFrame([dict(r) for r in store.query("SELECT label, lat, lon FROM places WHERE kind='address'")])
SITES = addr.sample(min(10, len(addr)), random_state=7) if len(addr) else pd.DataFrame(columns=['label', 'lat', 'lon'])
# SITES = pd.read_csv('my_sites.csv')   # label, lat, lon

def exposure(site):
    if not len(rated):
        return pd.Series({'High ≤ r': 0, 'Moderate ≤ r': 0, 'nearest High (km)': None})
    d = rated.apply(lambda r: haversine_m(site.lat, site.lon, r.lat, r.lon) / 1000, axis=1)
    near = rated[d <= NEARBY_KM]
    hi = d[rated.rating == 'High']
    return pd.Series({'High ≤ r': int((near.rating == 'High').sum()), 'Moderate ≤ r': int((near.rating == 'Moderate').sum()),
                      'nearest High (km)': round(hi.min(), 2) if len(hi) else None})

if len(SITES):
    exp = pd.concat([SITES.reset_index(drop=True), SITES.reset_index(drop=True).apply(exposure, axis=1)], axis=1)
    exp['status'] = exp.apply(lambda r: 'Exposed' if r['High ≤ r'] else ('Watch' if r['Moderate ≤ r'] else 'No flagged roads'), axis=1)
    print(f"r = {NEARBY_KM} km")
    display(exp.sort_values(['High ≤ r', 'Moderate ≤ r'], ascending=False)[['label', 'status', 'High ≤ r', 'Moderate ≤ r', 'nearest High (km)']])

## 4. Residents: the alert a neighbor would actually read

Type part of an address; suggestions only come from inside the imagery. The alert names the High and Moderate intersections
within 1.5 km, nearest first, in plain language. Set `USE_MODEL = True` to have the language model write it; otherwise a fixed template is used.

In [ ]:
QUERY = 'Clay Road'          # try part of any address in the covered neighborhoods
for s in suggest(store, QUERY):
    print(f"{s['label']}  ({s['kind']})")

In [ ]:
USE_MODEL = False
pick = suggest(store, QUERY) or [{'label': 'pin at 29.8459, -95.6383', 'lat': 29.8459, 'lon': -95.6383}]  # fallback if the address index is not built (run fetch-addresses)
if pick:
    p = pick[0]
    client = None
    if USE_MODEL:
        from flood_alert.vlm import VLMClient
        client = VLMClient(S, model=S.alert_model or None)
    a = build_alert(S, store, p['lat'], p['lon'], location_label=p['label'], client=client)
    print(a['text'])
    for i in a.get('named_intersections', []):
        print(f"  • {i['name']}: {i['rating']}, {i['distance_km']:.1f} km {i['direction']}")

## What this is and is not

- **Evidence, not a forecast.** Ratings say where water reached the road after Harvey, from imagery taken days later. A road that drained fast can show Low.
- **Model-made labels, spot-checked.** A vision model (qwen3-vl-8b) judged each crop; we hand-checked about 50 intersections on the review page. It misses floods (trees hide roads) more often than it invents them.
- **Coverage is the imagery.** Outside the 575 footprints an intersection is Unrated, never Low.
- **Blank FEMA never means safe.** With no flood-zone data and no visible evidence, an intersection is Moderate.
- **Nothing is sent.** Alerts are shown on screen; a real deployment would plug into the city's existing alert channel.